# MediFlow 공통 데이터셋 검증

대상만 선택하여 Hair(두피 현미경), Web Skin(얼굴 웹캠), Skin(피부 현미경)에 사용합니다.
각 대상은 별도 모델이며 클래스 순서는 기존 결과 JSON에서 가져왔습니다.
이 노트북 하나에 실행 코드가 포함되어 있어 별도 Python 파일 업로드가 필요 없습니다.

**사용 순서:** 설정 수정 → Colab GPU 선택(학습 시) → 위에서부터 모두 실행.
검증은 CPU로 가능합니다. 최초 설치 후 버전 확인에서 멈추면 세션을 다시 시작하고 처음부터 실행하세요.
이 코드는 새 실험용입니다. 과거 성능을 재현했다고 간주하지 않습니다.
기존 결과는 그대로 두고 실행마다 새 폴더에 기록합니다.


## 1. 사용자 설정 — 보통 여기만 수정합니다

`DOMAIN`: `hair`, `web_skin`, `skin` 중 하나입니다.
`PROJECT_ROOT`: Drive 프로젝트 폴더입니다. 이름을 바꿨다면 이 경로를 수정하세요.
`DATA_ZIP`: 비우면 datasets 아래에서 대상 이름으로 시작하는 ZIP을 찾습니다.
확장자가 없는 ZIP도 찾으며, 여러 개면 목록을 보여주고 멈춥니다. 그때 전체 경로를 입력하세요.
`AUDIT_DIR`: ① 검증이 완료된 결과 폴더입니다.
`EXPECTED_DATA_SHA256`: 별도 검증을 생략한 경우에만 확인된 ZIP 해시를 입력합니다.
학습 시 `AUDIT_DIR`과 `EXPECTED_DATA_SHA256` 중 하나가 필요합니다.
`RESUME_DIR`: 중단된 이번 공통 노트북 결과 폴더. 비우면 새 실행입니다.
완료 실험은 코드·설정·파일 일치 확인 후 재사용하며, 중단된 실험은 처음부터 다시 합니다.
과거 개별 노트북의 결과 폴더는 재개 대상으로 사용할 수 없습니다.


In [ ]:
DOMAIN = 'web_skin'
PROJECT_ROOT = '/content/drive/MyDrive/mediflow_Project'
DATA_ZIP = ''
AUDIT_DIR = ''
EXPECTED_DATA_SHA256 = ''
RESUME_DIR = ''
MODE = 'audit'
SEED = 42
BATCH_SIZE = 32
# 두 비교 실험은 모두 15회 head-only 학습. suite는 15회 + 미세조정 10회.
STAGE1_EPOCHS = 15
STAGE2_EPOCHS = 10
EXTENSION_EPOCHS = 5
# suite의 학습 데이터 종류. 원본·증강 비교에서는 자동으로 두 종류를 사용합니다.
TRAIN_VARIANT = 'augmented'


## 2. Drive 연결과 실행 환경

학습에는 GPU가 필요합니다. 메모리 부족 시 배치 크기를 16으로 낮추되 새 실험으로 시작하세요.
배치 크기 변경도 결과에 영향을 줄 수 있어 기록됩니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip -q install tensorflow==2.20.0 keras==3.13.2 pandas matplotlib pillow tqdm
import tensorflow as tf
import keras
if tf.__version__ != '2.20.0' or keras.__version__ != '3.13.2':
    raise RuntimeError('Colab 세션을 다시 시작한 뒤 처음부터 실행하세요.')
if MODE != 'audit' and not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('런타임 유형에서 GPU를 선택하세요.')


## 3. 공통 실행 코드 — 직접 수정할 필요 없습니다

결과에 이 소스와 생성 당시 Git commit을 함께 보관합니다. 미커밋 변경을 포함한 실제
실행 코드는 소스 사본과 SHA-256(파일 내용 식별값)으로 남깁니다.


In [ ]:
import sys, types
SOURCES = {'common_engine': '"""Sequential domain-configured experiments; validation selection precedes any test inference.\n\nThe Colab notebook embeds an exact copy of this module so no repository checkout\nis needed in Colab. Completed trials are reused only after artifact verification.\nInterrupted attempts are preserved and restarted, not resumed mid-epoch.\n"""\n\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport time\nimport uuid\nfrom pathlib import Path\n\nimport keras\nimport numpy as np\n\nPROTOCOL = "mediflow_common_v1"\nTRIALS = [\n    {"id": "b0_224_ce", "backbone": "B0", "size": 224, "loss": "ce"},\n    {"id": "b0_256_ce", "backbone": "B0", "size": 256, "loss": "ce"},\n    {"id": "b0_256_ls005", "backbone": "B0", "size": 256, "loss": "ls005"},\n    {"id": "b0_256_focal15", "backbone": "B0", "size": 256, "loss": "focal15"},\n    {"id": "b1_256_ls005", "backbone": "B1", "size": 256, "loss": "ls005"},\n]\n\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    temporary = path.with_name(".json-" + uuid.uuid4().hex[:12] + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef classification_metrics(truth, probabilities, count=5):\n    truth = np.asarray(truth, dtype=np.int64)\n    probabilities = np.asarray(probabilities)\n    if (\n        truth.ndim != 1\n        or not len(truth)\n        or probabilities.shape != (len(truth), count)\n        or not np.isfinite(probabilities).all()\n        or np.any(truth < 0)\n        or np.any(truth >= count)\n    ):\n        raise ValueError("Invalid evaluation arrays")\n    predictions = probabilities.argmax(axis=1)\n    cm = np.bincount(count * truth + predictions, minlength=count * count).reshape(count, count)\n    tp = np.diag(cm).astype(float)\n    precision = np.divide(tp, cm.sum(0), out=np.zeros(count), where=cm.sum(0) != 0)\n    recall = np.divide(tp, cm.sum(1), out=np.zeros(count), where=cm.sum(1) != 0)\n    f1 = np.divide(\n        2 * precision * recall,\n        precision + recall,\n        out=np.zeros(count),\n        where=precision + recall != 0,\n    )\n    return {\n        "accuracy": float(np.mean(truth == predictions)),\n        "macro_f1": float(f1.mean()),\n        "class_f1": f1.tolist(),\n        "precision": precision.tolist(),\n        "recall": recall.tolist(),\n        "support": cm.sum(1).tolist(),\n        "confusion_matrix": cm.tolist(),\n        "count": len(truth),\n    }\n\n\ndef predict_dataset(model, dataset):\n    truth, probabilities = [], []\n    for images, labels in dataset:\n        probabilities.extend(model(images, training=False).numpy())\n        truth.extend(np.argmax(labels.numpy(), axis=1))\n    return np.asarray(truth, dtype=np.int64), np.asarray(probabilities)\n\n\ndef save_predictions(path, paths, truth, probabilities):\n    if len(paths) != len(truth):\n        raise ValueError("File order and prediction count differ")\n    with Path(path).open("w", newline="", encoding="utf-8-sig") as stream:\n        writer = csv.writer(stream)\n        writer.writerow(\n            [\n                "path",\n                "true_index",\n                "pred_index",\n                *[f"prob_C{i}" for i in range(probabilities.shape[1])],\n            ]\n        )\n        for name, target, probs in zip(paths, truth, probabilities, strict=True):\n            writer.writerow([name, int(target), int(probs.argmax()), *map(float, probs)])\n\n\ndef loss_function(name):\n    if name == "ce":\n        return keras.losses.CategoricalCrossentropy()\n    if name == "ls005":\n        return keras.losses.CategoricalCrossentropy(label_smoothing=0.05)\n    if name == "focal15":\n        return keras.losses.CategoricalFocalCrossentropy(alpha=1.0, gamma=1.5)\n    raise ValueError(name)\n\n\ndef build_model(spec):\n    builder = {\n        "B0": keras.applications.EfficientNetB0,\n        "B1": keras.applications.EfficientNetB1,\n        "V2S": keras.applications.EfficientNetV2S,\n    }\n    size = spec["size"]\n    backbone = builder[spec["backbone"]](\n        include_top=False, weights="imagenet", input_shape=(size, size, 3)\n    )\n    backbone.trainable = False\n    inputs = keras.Input((size, size, 3))\n    features = backbone(inputs, training=False)\n    features = keras.layers.GlobalAveragePooling2D()(features)\n    features = keras.layers.Dropout(0.3)(features)\n    outputs = keras.layers.Dense(spec["class_count"], activation="softmax")(features)\n    model = keras.Model(inputs, outputs)\n    model.compile(\n        optimizer=keras.optimizers.Adam(1e-4),\n        loss=loss_function(spec["loss"]),\n        metrics=["accuracy"],\n    )\n    return model\n\n\ndef configure_partial(model, loss_name):\n    backbones = [\n        layer\n        for layer in model.layers\n        if isinstance(layer, keras.Model) and "efficientnet" in layer.name.lower()\n    ]\n    if len(backbones) != 1:\n        raise ValueError("Expected one EfficientNet backbone")\n    backbone = backbones[0]\n    backbone.trainable = True\n    for index, layer in enumerate(backbone.layers):\n        layer.trainable = index >= len(backbone.layers) - 30 and not isinstance(\n            layer, keras.layers.BatchNormalization\n        )\n    model.compile(\n        optimizer=keras.optimizers.Adam(1e-5), loss=loss_function(loss_name), metrics=["accuracy"]\n    )\n    return [layer.name for layer in backbone.layers if layer.trainable]\n\n\nclass HistoryBackup(keras.callbacks.Callback):\n    def __init__(self, path):\n        super().__init__()\n        self.path = path\n        self.values = {}\n\n    def on_epoch_end(self, epoch, logs=None):\n        for key, value in (logs or {}).items():\n            self.values.setdefault(key, []).append(float(value))\n        write_json(self.path, self.values)\n\n\ndef fit_stage(model, train, val, directory, name, epochs):\n    best = directory / (name + "_best.keras")\n    callbacks = [\n        keras.callbacks.ModelCheckpoint(\n            str(best), monitor="val_accuracy", mode="max", save_best_only=True\n        ),\n        keras.callbacks.CSVLogger(str(directory / (name + "_log.csv"))),\n        HistoryBackup(directory / (name + "_history.json")),\n        keras.callbacks.TerminateOnNaN(),\n    ]\n    history = model.fit(train, validation_data=val, epochs=epochs, callbacks=callbacks, verbose=2)\n    values = {key: [float(v) for v in seq] for key, seq in history.history.items()}\n    if len(values.get("val_accuracy", [])) != epochs or not all(\n        np.isfinite(seq).all() for seq in values.values()\n    ):\n        raise RuntimeError("Incomplete or non-finite training; attempt retained")\n    model.save(directory / (name + "_last.keras"))\n    return values, best\n\n\ndef checkpoint_choice(baseline_score, new_score):\n    """Keep the earlier/simpler checkpoint on ties."""\n    return new_score > baseline_score\n\n\ndef cached_record(root, trial_id, signature):\n    trial_dir = Path(root) / trial_id\n    marker = trial_dir / "completed.json"\n    if not marker.exists():\n        return None\n    record = read_json(marker)\n    if record["signature"] != signature:\n        raise ValueError("Resume settings differ; use a new suite directory")\n    for relative, digest in record["artifact_hashes"].items():\n        target = (trial_dir / relative).resolve()\n        if not target.is_relative_to(trial_dir.resolve()) or file_hash(target) != digest:\n            raise ValueError("Completed artifact changed or corrupted: " + relative)\n    return record\n\n\ndef evaluate_to_files(model, dataset, paths, directory, prefix):\n    truth, probabilities = predict_dataset(model, dataset)\n    metrics = classification_metrics(truth, probabilities, probabilities.shape[1])\n    write_json(directory / (prefix + "_metrics.json"), metrics)\n    save_predictions(directory / (prefix + "_predictions.csv"), paths, truth, probabilities)\n    return metrics\n\n\ndef run_trial(spec, dataset_factory, root, signature, seed=42, epochs1=15, epochs2=10):\n    cached = cached_record(root, spec["id"], signature)\n    if cached:\n        return cached\n    keras.backend.clear_session()\n    keras.utils.set_random_seed(seed)\n    directory = Path(root) / spec["id"] / ("attempt_" + uuid.uuid4().hex[:12])\n    directory.mkdir(parents=True, exist_ok=False)\n    write_json(directory / "spec.json", spec)\n    train, _ = dataset_factory("train", spec["size"], True)\n    val, paths = dataset_factory("val", spec["size"], False)\n    started = time.monotonic()\n    model = build_model(spec)\n    h1, best1 = fit_stage(model, train, val, directory, "stage1", epochs1)\n    del model\n    keras.backend.clear_session()\n    model = keras.models.load_model(best1, compile=False)\n    trainable = []\n    h2 = {key: [] for key in h1}\n    best2 = best1\n    if epochs2:\n        trainable = configure_partial(model, spec["loss"])\n        h2, best2 = fit_stage(model, train, val, directory, "stage2", epochs2)\n    del model\n    selected_stage = (\n        "stage2"\n        if checkpoint_choice(max(h1["val_accuracy"]), max(h2["val_accuracy"], default=-1.0))\n        else "stage1"\n    )\n    selected = best2 if selected_stage == "stage2" else best1\n    model = keras.models.load_model(selected, compile=False)\n    metrics = evaluate_to_files(model, val, paths, directory, "validation")\n    record = {\n        "id": spec["id"],\n        "spec": spec,\n        "signature": signature,\n        "attempt": directory.name,\n        "selected_model": selected.name,\n        "selected_stage": selected_stage,\n        "validation": metrics,\n        "stage1_best_val": max(h1["val_accuracy"]),\n        "stage2_best_val": max(h2["val_accuracy"]) if h2["val_accuracy"] else None,\n        "training_seconds": time.monotonic() - started,\n        "parameters": model.count_params(),\n        "model_bytes": selected.stat().st_size,\n        "trainable_backbone_layers": trainable,\n        "history": {key: h1[key] + h2[key] for key in h1},\n        "stage_boundary": len(h1["accuracy"]),\n    }\n    finish_record(directory, record)\n    return record\n\n\ndef finish_record(directory, record):\n    write_json(directory / "record.json", record)\n    record["artifact_hashes"] = {\n        str(path.relative_to(directory.parent)): file_hash(path)\n        for path in directory.iterdir()\n        if path.is_file()\n    }\n    write_json(directory.parent / "completed.json", record)\n\n\ndef selected_model_path(root, record):\n    return Path(root) / record["id"] / record["attempt"] / record["selected_model"]\n\n\ndef extend_b1(parent, dataset_factory, root, signature, seed=42, epochs=5):\n    trial_id = "b1_256_ls005_extend5"\n    cached = cached_record(root, trial_id, signature)\n    if cached:\n        return cached\n    keras.backend.clear_session()\n    keras.utils.set_random_seed(seed)\n    directory = Path(root) / trial_id / ("attempt_" + uuid.uuid4().hex[:12])\n    directory.mkdir(parents=True, exist_ok=False)\n    parent_dir = Path(root) / parent["id"] / parent["attempt"]\n    # Continue from epoch 10\'s LAST checkpoint, including optimizer state.\n    source = parent_dir / "stage2_last.keras"\n    model = keras.models.load_model(source)\n    if model.optimizer is None:\n        raise ValueError("Extension requires saved optimizer")\n    train, _ = dataset_factory("train", 256, True)\n    val, paths = dataset_factory("val", 256, False)\n    started = time.monotonic()\n    history, best = fit_stage(model, train, val, directory, "extension", epochs)\n    del model\n    parent_best = selected_model_path(root, parent)\n    parent_score = max(parent["stage1_best_val"], parent["stage2_best_val"])\n    keep_extension = checkpoint_choice(parent_score, max(history["val_accuracy"]))\n    selected = directory / "selected.keras"\n    import shutil\n\n    shutil.copyfile(best if keep_extension else parent_best, selected)\n    model = keras.models.load_model(selected, compile=False)\n    metrics = evaluate_to_files(model, val, paths, directory, "validation")\n    record = {\n        "id": trial_id,\n        "spec": {**parent["spec"], "id": trial_id},\n        "signature": signature,\n        "attempt": directory.name,\n        "selected_model": selected.name,\n        "selected_stage": "extension" if keep_extension else "parent_" + parent["selected_stage"],\n        "validation": metrics,\n        "parameters": model.count_params(),\n        "model_bytes": selected.stat().st_size,\n        "training_seconds": time.monotonic() - started,\n        "history": {key: parent["history"][key] + history[key] for key in history},\n        "stage_boundary": parent["stage_boundary"],\n        "extension_boundary": len(parent["history"]["accuracy"]),\n        "parent_last_sha256": file_hash(source),\n        "optimizer_restored": True,\n        "extension_best_val": max(history["val_accuracy"]),\n        "stage1_best_val": parent["stage1_best_val"],\n        "stage2_best_val": parent["stage2_best_val"],\n    }\n    finish_record(directory, record)\n    return record\n\n\ndef select_winner(records):\n    # Stable order preserves earlier experiments on exact ties.\n    return max(records, key=lambda record: record["validation"]["accuracy"])\n', 'common_audit': '"""Original Web Skin mechanical audit generalized to three class contracts.\n\nByte/pixel hashes do not establish person, lesion, session or augmentation lineage.\n"""\n\nimport hashlib\nfrom collections import Counter\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport pandas as pd\nfrom IPython.display import display\nfrom PIL import Image\nfrom tqdm.auto import tqdm\n\nfrom mediflow_datasets.common_engine import file_hash, write_json\n\n\ndef audit_dataset(extract_root, report_dir, domain, classes, data_hash):\n    EXTRACT_ROOT, REPORT_DIR = Path(extract_root), Path(report_dir)\n    CLASS_NAMES = classes\n    CLASS_CODES = {name: f"C{i}" for i, name in enumerate(classes)}\n    DATA_SHA256 = data_hash\n    SPLITS = ("train", "val", "test")\n    IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".gif", ".tif", ".tiff"}\n    TRAIN_LOADER_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}\n    sha256 = file_hash\n\n    def save_json(name, value):\n        write_json(REPORT_DIR / name, value)\n\n    def save_csv(name, rows, columns):\n        pd.DataFrame(rows, columns=columns).to_csv(\n            REPORT_DIR / name, index=False, encoding="utf-8-sig"\n        )\n\n    roots, inventory, issues, metadata_files = {}, [], [], []\n    for path in EXTRACT_ROOT.rglob("*"):\n        if path.is_file() and path.suffix.lower() in {".json", ".csv"}:\n            metadata_files.append(str(path.relative_to(EXTRACT_ROOT)))\n    for kind in ("original", "augmented"):\n        matches = [\n            p\n            for p in EXTRACT_ROOT.rglob("*")\n            if p.is_dir()\n            and p.name.lower() == kind\n            and all((p / split).is_dir() for split in SPLITS)\n        ]\n        if len(matches) != 1:\n            issues.append(\n                {"type": "root_structure", "detail": f"{kind}: {[str(p) for p in matches]}"}\n            )\n            continue\n        root = roots[kind] = matches[0]\n        for split in SPLITS:\n            classes = sorted(p.name for p in (root / split).iterdir() if p.is_dir())\n            if classes != sorted(CLASS_NAMES):\n                issues.append({"type": "class_structure", "detail": f"{kind}/{split}: {classes}"})\n            stray = [\n                p\n                for p in (root / split).iterdir()\n                if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS\n            ]\n            if stray:\n                issues.append(\n                    {"type": "image_outside_class", "detail": f"{kind}/{split}: {len(stray)}"}\n                )\n            for cls in sorted(set(classes) | set(CLASS_NAMES)):\n                paths = sorted(\n                    p\n                    for p in (root / split / cls).rglob("*")\n                    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS\n                )\n                if not paths:\n                    issues.append({"type": "empty_class", "detail": f"{kind}/{split}/{cls}"})\n                for path in tqdm(paths, desc=f"{kind}/{split}/{CLASS_CODES.get(cls, cls)}"):\n                    relative_path = str(path.relative_to(EXTRACT_ROOT))\n                    row = {"kind": kind, "split": split, "class_name": cls, "path": relative_path}\n                    try:\n                        row["sha256"] = sha256(path)\n                        with Image.open(path) as image:\n                            row.update(\n                                width=image.width,\n                                height=image.height,\n                                mode=image.mode,\n                                frames=getattr(image, "n_frames", 1),\n                            )\n                            rgb = image.convert("RGB")\n                            row["pixel_sha256"] = hashlib.sha256(\n                                str(rgb.size).encode("ascii") + b":" + rgb.tobytes()\n                            ).hexdigest()\n                        row["readable"] = True\n                        if row["frames"] != 1:\n                            issues.append({"type": "multi_frame_image", "detail": relative_path})\n                    except Exception as exc:\n                        row["readable"] = False\n                        issues.append(\n                            {"type": "unreadable_image", "detail": f"{relative_path}: {exc}"}\n                        )\n                    if path.suffix.lower() not in TRAIN_LOADER_EXTENSIONS:\n                        issues.append(\n                            {"type": "training_loader_unsupported", "detail": relative_path}\n                        )\n                    inventory.append(row)\n    df = pd.DataFrame(\n        inventory,\n        columns=[\n            "kind",\n            "split",\n            "class_name",\n            "path",\n            "sha256",\n            "pixel_sha256",\n            "width",\n            "height",\n            "mode",\n            "frames",\n            "readable",\n        ],\n    )\n    df.to_csv(REPORT_DIR / "image_inventory.csv", index=False, encoding="utf-8-sig")\n    save_json(\n        "metadata_inventory.json",\n        {\n            "files": sorted(metadata_files),\n            "status": "파일 목록만 수집. 사람·병변·촬영 세션·증강 출처 대응 관계는 미검증",\n        },\n    )\n    print("조사한 이미지 파일 수:", len(df))\n\n    cross_split_rows, conflict_rows, within_rows, pair_rows, compare_rows = [], [], [], [], []\n    for key in ("sha256", "pixel_sha256"):\n        valid = df.dropna(subset=[key])\n        for digest, group in valid.groupby(key):\n            if group["split"].nunique() > 1:\n                cross_split_rows.append(\n                    {\n                        "hash_type": key,\n                        "hash": digest,\n                        "file_count": len(group),\n                        "splits": "|".join(sorted(group["split"].unique())),\n                        "paths": "|".join(group["path"]),\n                    }\n                )\n            if group["class_name"].nunique() > 1:\n                conflict_rows.append(\n                    {\n                        "hash_type": key,\n                        "hash": digest,\n                        "file_count": len(group),\n                        "classes": "|".join(sorted(group["class_name"].unique())),\n                        "paths": "|".join(group["path"]),\n                    }\n                )\n        for (kind, split, digest), group in valid.groupby(["kind", "split", key]):\n            if len(group) > 1:\n                within_rows.append(\n                    {\n                        "kind": kind,\n                        "split": split,\n                        "hash_type": key,\n                        "hash": digest,\n                        "file_count": len(group),\n                        "paths": "|".join(group["path"]),\n                    }\n                )\n        for kind in ("original", "augmented"):\n            for left, right in (("train", "val"), ("train", "test"), ("val", "test")):\n                a = set(valid[(valid.kind == kind) & (valid.split == left)][key])\n                b = set(valid[(valid.kind == kind) & (valid.split == right)][key])\n                pair_rows.append(\n                    {\n                        "kind": kind,\n                        "hash_type": key,\n                        "pair": left + "_" + right,\n                        "shared_hash_groups": len(a & b),\n                    }\n                )\n\n    def counted(kind, split, key="sha256"):\n        part = df[(df.kind == kind) & (df.split == split)].dropna(subset=[key])\n        return Counter(zip(part["class_name"], part[key], strict=True))\n\n    if len(roots) == 2:\n        for split in ("val", "test"):\n            a, b = counted("original", split), counted("augmented", split)\n            compare_rows.append(\n                {\n                    "check": "evaluation_multiset_equal",\n                    "split": split,\n                    "passed": a == b,\n                    "missing": sum((a - b).values()),\n                    "extra": sum((b - a).values()),\n                }\n            )\n            if a != b:\n                issues.append({"type": "evaluation_mismatch", "detail": split})\n        missing = counted("original", "train") - counted("augmented", "train")\n        compare_rows.append(\n            {\n                "check": "original_train_contained",\n                "split": "train",\n                "passed": not missing,\n                "missing": sum(missing.values()),\n                "extra": None,\n            }\n        )\n        if missing:\n            issues.append({"type": "missing_train_original", "detail": str(sum(missing.values()))})\n    if cross_split_rows:\n        issues.append(\n            {"type": "cross_split_duplicates", "detail": "cross_split_duplicates.csv 참조"}\n        )\n    if conflict_rows:\n        issues.append({"type": "label_conflicts", "detail": "label_conflicts.csv 참조"})\n\n    save_csv(\n        "cross_split_duplicates.csv",\n        cross_split_rows,\n        ["hash_type", "hash", "file_count", "splits", "paths"],\n    )\n    save_csv(\n        "label_conflicts.csv",\n        conflict_rows,\n        ["hash_type", "hash", "file_count", "classes", "paths"],\n    )\n    save_csv(\n        "within_split_duplicates.csv",\n        within_rows,\n        ["kind", "split", "hash_type", "hash", "file_count", "paths"],\n    )\n    save_csv(\n        "split_overlap_counts.csv", pair_rows, ["kind", "hash_type", "pair", "shared_hash_groups"]\n    )\n    save_csv(\n        "dataset_consistency.csv", compare_rows, ["check", "split", "passed", "missing", "extra"]\n    )\n    save_csv("audit_issues.csv", issues, ["type", "detail"])\n    counts = df.groupby(["kind", "split", "class_name"]).size().rename("count").reset_index()\n    counts.to_csv(REPORT_DIR / "dataset_counts.csv", index=False, encoding="utf-8-sig")\n    resolution = (\n        df.groupby(["kind", "split", "width", "height"]).size().rename("count").reset_index()\n    )\n    resolution.to_csv(REPORT_DIR / "resolution_counts.csv", index=False)\n    summary = {\n        "domain": domain,\n        "data_sha256": DATA_SHA256,\n        "class_names": CLASS_NAMES,\n        "status": "issues_found" if issues else "mechanical_checks_passed_with_limitations",\n        "issue_count": len(issues),\n        "image_files": len(df),\n        "counts": counts.to_dict("records"),\n        "cross_split_groups_by_hash_type": dict(Counter(r["hash_type"] for r in cross_split_rows)),\n        "label_conflict_groups_by_hash_type": dict(Counter(r["hash_type"] for r in conflict_rows)),\n        "within_split_duplicate_groups_by_hash_type": dict(\n            Counter(r["hash_type"] for r in within_rows)\n        ),\n        "metadata_file_count": len(metadata_files),\n        "limitations": [\n            "사람·병변·촬영 세션의 이미지 대응 정보 미검증",\n            "증강 출처 기록의 연결 미검증; 변환된 파생본은 해시가 다를 수 있음",\n            "재압축·밝기·회전·유사 장면 탐지는 미수행",\n            "폴더 라벨의 의미상 정확성 및 복수 상태 동시 존재 여부 미검증",\n            "이전 학습 당시 데이터 해시가 없어 원 학습 분할 동일성 입증 불가",\n            "실제 대상 장비 데이터 검증은 별도",\n        ],\n        "interpretation": (\n            "SHA와 pixel 그룹은 같은 사례가 겹칠 수 있음. "\n            "Original/Augmented 복사본도 포함하므로 합산 제거 수로 사용 금지."\n        ),\n        "protocol": "common_audit_v1",\n        "next_step": "보고서 검토 후 정제 또는 기존 모델 재현 여부 결정. 자동 학습하지 않음.",\n    }\n    save_json("audit_summary.json", summary)\n    display(counts)\n    display(pd.DataFrame(pair_rows))\n    display(pd.DataFrame(compare_rows))\n    print("검사 상태:", summary["status"], "문제 항목:", len(issues))\n    print("\\n".join(summary["limitations"]))\n\n    pairs, seen = [], set()\n    for key in ("sha256", "pixel_sha256"):\n        for _digest, group in df.dropna(subset=[key]).groupby(key):\n            if group["split"].nunique() <= 1 and group["class_name"].nunique() <= 1:\n                continue\n            readable = group[group["readable"]]\n            if readable.empty:\n                continue\n            left = readable.iloc[0]\n            alternatives = readable[\n                (readable["split"] != left["split"])\n                | (readable["class_name"] != left["class_name"])\n            ]\n            if alternatives.empty:\n                continue\n            right = alternatives.iloc[0]\n            identity = tuple(sorted([left["path"], right["path"]]))\n            if identity in seen:\n                continue\n            seen.add(identity)\n            pairs.append((key, left, right))\n            if len(pairs) >= 8:\n                break\n        if len(pairs) >= 8:\n            break\n    example_rows = []\n    if pairs:\n        fig, axes = plt.subplots(len(pairs), 2, figsize=(10, 3.6 * len(pairs)), squeeze=False)\n        for index, (key, left, right) in enumerate(pairs):\n            example_rows.append(\n                {\n                    "case": index + 1,\n                    "hash_type": key,\n                    "left_path": left["path"],\n                    "right_path": right["path"],\n                }\n            )\n            for side, row in enumerate((left, right)):\n                with Image.open(EXTRACT_ROOT / row["path"]) as source:\n                    axes[index, side].imshow(source.convert("RGB"))\n                label = CLASS_CODES.get(row["class_name"], "unexpected class")\n                axes[index, side].set_title(\n                    f"Case {index + 1}: {row[\'kind\']}/{row[\'split\']}/{label}\\n{key}", fontsize=9\n                )\n                axes[index, side].axis("off")\n        fig.tight_layout()\n        fig.savefig(REPORT_DIR / "duplicate_examples.png", dpi=140)\n        plt.show()\n        plt.close(fig)\n    else:\n        print("표시할 분할 간 중복/라벨 충돌 사진 쌍이 없습니다.")\n    save_csv("example_pairs.csv", example_rows, ["case", "hash_type", "left_path", "right_path"])\n    if not counts.empty:\n        plot_counts = counts.copy()\n        plot_counts["class_name"] = plot_counts["class_name"].map(\n            lambda x: CLASS_CODES.get(x, "unexpected")\n        )\n        fig, axes = plt.subplots(1, 2, figsize=(13, 5))\n        for ax, kind in zip(axes, ("original", "augmented"), strict=True):\n            part = plot_counts[plot_counts.kind == kind]\n            if not part.empty:\n                part.pivot_table(\n                    index="class_name", columns="split", values="count", aggfunc="sum", fill_value=0\n                ).plot.bar(ax=ax)\n            ax.set(title=domain + " " + kind, ylabel="Image count", xlabel="Class code")\n        fig.tight_layout()\n        fig.savefig(REPORT_DIR / "dataset_counts.png", dpi=180)\n        plt.show()\n        plt.close(fig)\n    save_json("class_code_mapping.json", CLASS_CODES)\n    print(CLASS_CODES)\n\n    return summary\n', 'common_workflow': '"""Standalone Colab orchestration; shared contracts, audit gates and artifacts."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport platform\nimport shutil\nimport stat\nimport uuid\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path, PurePosixPath\n\nimport keras\nimport numpy as np\nimport tensorflow as tf\n\nfrom mediflow_datasets import common_engine as engine\n\nEXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}\n\n\ndef signature(value):\n    return hashlib.sha256(\n        json.dumps(value, sort_keys=True, ensure_ascii=False).encode()\n    ).hexdigest()\n\n\ndef extract_zip(source, destination):\n    """Validate every member before creating any output; no overwrite."""\n    destination = Path(destination).resolve()\n    if destination.exists():\n        raise FileExistsError(destination)\n    with zipfile.ZipFile(source) as archive:\n        seen = set()\n        for item in archive.infolist():\n            name = item.orig_filename\n            path = PurePosixPath(name)\n            if (\n                path.is_absolute()\n                or ".." in path.parts\n                or "\\\\" in name\n                or ":" in name\n                or stat.S_ISLNK(item.external_attr >> 16)\n            ):\n                raise ValueError("Unsafe ZIP member: " + name)\n            key = name.rstrip("/").casefold()\n            if key in seen:\n                raise ValueError("Duplicate ZIP destination: " + name)\n            seen.add(key)\n        destination.mkdir(parents=True)\n        archive.extractall(destination)\n\n\ndef roots_for(extracted):\n    roots = {}\n    for kind in ("original", "augmented"):\n        matches = [\n            p\n            for p in Path(extracted).rglob("*")\n            if p.is_dir()\n            and p.name.lower() == kind\n            and all((p / s).is_dir() for s in ("train", "val", "test"))\n        ]\n        if len(matches) != 1:\n            raise ValueError(f"{kind}/train,val,test 구조를 하나로 확인하세요: {matches}")\n        roots[kind] = matches[0]\n    return roots\n\n\ndef verify_audit(directory, domain, classes, digest):\n    directory = Path(directory)\n    manifest = engine.read_json(directory / "audit_manifest.json")\n    for name in ("audit_summary.json", "image_inventory.csv", "audit_issues.csv"):\n        if engine.file_hash(directory / name) != manifest[name]:\n            raise ValueError("검증 보고서가 변경됐습니다: " + name)\n    audit = engine.read_json(directory / "audit_summary.json")\n    if (\n        audit["domain"] != domain\n        or audit["class_names"] != classes\n        or audit["data_sha256"] != digest\n        or audit.get("protocol") != "common_audit_v1"\n        or audit["status"] != "mechanical_checks_passed_with_limitations"\n    ):\n        raise ValueError("대상/클래스/데이터가 다르거나 검증 문제가 있습니다. 공통 ①을 확인하세요.")\n    return audit\n\n\ndef prepare(config, profiles, sources, commit, local_parent="/content"):\n    c = dict(config)\n    c.setdefault("expected_data_sha256", "")\n    c.setdefault("seeds", [c["seed"]])\n    if c["domain"] not in profiles or c["mode"] not in (\n        "audit",\n        "comparison",\n        "suite",\n        "baseline3",\n        "paper_suite",\n        "supcon_compare",\n        "supcon_repeat",\n        "paper_screen",\n        "sam_screen",\n        "final_candidate",\n        "web_skin_wsdan",\n        "web_skin_paper_suite",\n        "web_skin_pmg_b1_384",\n        "web_skin_pmg_final",\n    ):\n        raise ValueError("DOMAIN/MODE 설정을 확인하세요.")\n    if c["train_variant"] not in ("original", "augmented"):\n        raise ValueError("TRAIN_VARIANT는 original 또는 augmented입니다.")\n    for key in ("batch_size", "epochs1", "epochs2", "extension_epochs"):\n        if not isinstance(c[key], int) or c[key] <= 0:\n            raise ValueError(key + "는 양의 정수여야 합니다.")\n    if (\n        not isinstance(c["seeds"], list)\n        or not c["seeds"]\n        or any(not isinstance(value, int) or value < 0 for value in c["seeds"])\n        or len(set(c["seeds"])) != len(c["seeds"])\n    ):\n        raise ValueError("SEEDS는 서로 다른 0 이상의 정수 목록이어야 합니다.")\n    if c["mode"] == "baseline3" and len(c["seeds"]) != 3:\n        raise ValueError("baseline3는 정확히 3개의 seed가 필요합니다.")\n    if c["mode"] == "paper_suite" and len(c["seeds"]) != 3:\n        raise ValueError("paper_suite는 정확히 3개의 seed가 필요합니다.")\n    if c["mode"] == "supcon_compare" and c["seeds"] != [42]:\n        raise ValueError("supcon_compare의 선별 seed는 [42]여야 합니다.")\n    if c["mode"] == "supcon_repeat" and c["seeds"] != [43, 44]:\n        raise ValueError("supcon_repeat의 확인 seed는 [43, 44]여야 합니다.")\n    if c["mode"] == "paper_screen" and c["seeds"] != [42]:\n        raise ValueError("paper_screen의 선별 seed는 [42]여야 합니다.")\n    if c["mode"] in (\n        "sam_screen",\n        "final_candidate",\n        "web_skin_pmg_final",\n    ) and c["seeds"] != [42]:\n        raise ValueError(c["mode"] + "의 seed는 [42]여야 합니다.")\n    project = Path(c["project_root"])\n    if not project.is_dir():\n        raise FileNotFoundError("PROJECT_ROOT 폴더를 확인하세요: " + str(project))\n    if c["mode"] != "audit" and not (\n        c["audit_dir"] or c["expected_data_sha256"]\n    ):\n        raise ValueError("AUDIT_DIR 또는 확인된 EXPECTED_DATA_SHA256을 입력하세요.")\n    if c["data_zip"]:\n        candidates = [Path(c["data_zip"])]\n    else:\n        candidates = sorted(\n            p\n            for p in (project / "datasets").rglob(c["domain"] + "*")\n            if p.is_file() and zipfile.is_zipfile(p)\n        )\n    if len(candidates) != 1 or not zipfile.is_zipfile(candidates[0]):\n        raise ValueError(f"DATA_ZIP으로 ZIP 하나를 지정하세요: {candidates}")\n    source = candidates[0]\n    run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]\n    local = Path(local_parent) / ("mediflow_" + run_id)\n    local.mkdir(parents=True, exist_ok=False)\n    with zipfile.ZipFile(source) as archive:\n        required = (\n            source.stat().st_size + sum(m.file_size for m in archive.infolist()) + 2 * 1024**3\n        )\n    if required > shutil.disk_usage(local).free:\n        raise RuntimeError("Colab 압축 해제 공간이 부족합니다.")\n    copied = local / "input.zip"\n    shutil.copyfile(source, copied)\n    digest = engine.file_hash(copied)\n    if digest != engine.file_hash(source):\n        raise OSError("Drive ZIP 복사 내용 불일치")\n    classes = profiles[c["domain"]]\n    audit = None\n    if c["mode"] != "audit":\n        if c["audit_dir"]:\n            audit = verify_audit(c["audit_dir"], c["domain"], classes, digest)\n        else:\n            expected = c["expected_data_sha256"].strip().lower()\n            if len(expected) != 64 or any(ch not in "0123456789abcdef" for ch in expected):\n                raise ValueError("EXPECTED_DATA_SHA256은 64자리 SHA-256이어야 합니다.")\n            if digest != expected:\n                raise ValueError("DATA_ZIP이 확인된 SHA-256과 다릅니다.")\n            audit = {\n                "domain": c["domain"],\n                "class_names": classes,\n                "data_sha256": digest,\n                "protocol": "expected_sha256_v1",\n                "status": "independent_audit_skipped",\n                "limitations": [\n                    "Independent common audit was skipped by the project owner",\n                    "Person, lesion and capture-session leakage remains unverified",\n                    "Perceptual near-duplicate and clinical label checks were not performed",\n                ],\n            }\n    extract_zip(copied, local / "dataset")\n    settings = {\n        k: c[k]\n        for k in (\n            "domain",\n            "mode",\n            "seed",\n            "seeds",\n            "batch_size",\n            "epochs1",\n            "epochs2",\n            "extension_epochs",\n            "train_variant",\n        )\n    }\n    settings.update(\n        classes=classes,\n        data_sha256=digest,\n        source_hashes={k: signature(v) for k, v in sources.items()},\n        protocol="common_v1",\n        audit=signature(audit),\n        environment=dict(\n            tensorflow=tf.__version__,\n            keras=keras.__version__,\n            numpy=np.__version__,\n            python=platform.python_version(),\n        ),\n    )\n    if c["mode"] in (\n        "paper_suite",\n        "supcon_compare",\n        "supcon_repeat",\n        "paper_screen",\n        "sam_screen",\n        "web_skin_wsdan",\n        "web_skin_paper_suite",\n        "web_skin_pmg_b1_384",\n    ):\n        settings["experiments"] = c.get("experiments", [])\n    if c["mode"] in ("sam_screen", "final_candidate", "web_skin_pmg_final"):\n        settings.update(\n            parent_run_dir=c.get("parent_run_dir"),\n            parent_model_sha256=c.get(\n                "parent_model_sha256", c.get("parent_stage1_sha256")\n            ),\n        )\n    if c["mode"] == "sam_screen":\n        settings["sam_rho"] = c.get("sam_rho")\n    if c["mode"] in ("web_skin_wsdan", "web_skin_paper_suite"):\n        settings.update(\n            attention_maps=c.get("attention_maps"),\n            crop_threshold=c.get("crop_threshold"),\n            drop_threshold=c.get("drop_threshold"),\n        )\n    if c["mode"] in ("web_skin_paper_suite", "web_skin_pmg_b1_384"):\n        settings.update(\n            pmg_jigsaw_grids=c.get("pmg_jigsaw_grids"),\n        )\n    if c["mode"] == "web_skin_paper_suite":\n        settings.update(\n            mixstyle_alpha=c.get("mixstyle_alpha"),\n            mixstyle_probability=c.get("mixstyle_probability"),\n        )\n    sig = signature(settings)\n    if c["resume_dir"]:\n        output = Path(c["resume_dir"])\n        if c["mode"] == "audit":\n            raise ValueError("검사는 새 실행으로 시작하세요. RESUME_DIR을 비우세요.")\n        if engine.read_json(output / "run_config.json")["signature"] != sig:\n            raise ValueError(\n                "코드/설정/환경/데이터/검증이 다른 실행입니다. 새 결과 폴더를 사용하세요."\n            )\n    else:\n        output = project / "2_results" / c["domain"] / (c["mode"] + "_" + run_id)\n        output.mkdir(parents=True, exist_ok=False)\n        engine.write_json(\n            output / "run_config.json",\n            {\n                "signature": sig,\n                "settings": settings,\n                "code_commit_at_generation": commit,\n                "code_state": "embedded sources include uncommitted changes; exact sources saved",\n                "source_zip": str(source),\n                "audit_source": c["audit_dir"] or "expected_data_sha256_only",\n                "baseline": (\n                    (\n                        "Fixed PMG B0/256 Validation candidate reused; model not retrained"\n                        if c["mode"] == "web_skin_pmg_final"\n                        else "Saved B0/256/CE Validation metrics reused; baseline not retrained"\n                    )\n                    if c["mode"]\n                    in (\n                        "web_skin_wsdan",\n                        "web_skin_paper_suite",\n                        "web_skin_pmg_b1_384",\n                        "web_skin_pmg_final",\n                    )\n                    else "ImageNet pretrained EfficientNet; historical metrics not reused"\n                ),\n                "gpu": [str(d) for d in tf.config.list_physical_devices("GPU")],\n            },\n        )\n        for name, code in sources.items():\n            (output / (name + ".py")).write_text(code, encoding="utf-8")\n        engine.write_json(output / "class_names.json", classes)\n        if c["audit_dir"]:\n            shutil.copyfile(\n                Path(c["audit_dir"]) / "audit_summary.json", output / "audit_summary.json"\n            )\n            shutil.copyfile(\n                Path(c["audit_dir"]) / "image_inventory.csv", output / "image_inventory.csv"\n            )\n        elif audit:\n            engine.write_json(output / "audit_summary.json", audit)\n    context = dict(\n        config=c,\n        classes=classes,\n        signature=sig,\n        output=output,\n        local=local,\n        data_hash=digest,\n        audit=audit,\n        extracted=local / "dataset",\n        project=project,\n    )\n    if c["mode"] != "audit":\n        context["roots"] = roots_for(context["extracted"])\n        # Dataset ZIP is immutable and matches the audit; verify class folders again.\n        for root in context["roots"].values():\n            for split in ("train", "val", "test"):\n                found = sorted(p.name for p in (root / split).iterdir() if p.is_dir())\n                if found != sorted(classes):\n                    raise ValueError(f"클래스 불일치: {root / split}")\n    print("실행 결과:", output)\n    return context\n\n\ndef archive_results(context):\n    output = context["output"]\n    destination = output.parent / (output.name + "_results_" + uuid.uuid4().hex[:8] + ".zip")\n    with zipfile.ZipFile(destination, "x", compression=zipfile.ZIP_DEFLATED) as archive:\n        for p in sorted(output.rglob("*")):\n            if p.is_file() and p.suffix not in (".keras", ".tmp"):\n                archive.write(p, output.name + "/" + p.relative_to(output).as_posix())\n    with zipfile.ZipFile(destination) as archive:\n        if archive.testzip():\n            raise OSError("보고서 ZIP 검사 실패")\n    print("로컬로 내려받을 결과 ZIP:", destination)\n    return destination\n\n\ndef audit_run(context):\n    from mediflow_datasets.common_audit import audit_dataset\n\n    summary = audit_dataset(\n        context["extracted"],\n        context["output"],\n        context["config"]["domain"],\n        context["classes"],\n        context["data_hash"],\n    )\n    engine.write_json(\n        context["output"] / "audit_manifest.json",\n        {\n            name: engine.file_hash(context["output"] / name)\n            for name in ("audit_summary.json", "image_inventory.csv", "audit_issues.csv")\n        },\n    )\n    archive_results(context)\n    print("검사 상태:", summary["status"], "\\n학습 AUDIT_DIR:", context["output"])\n    return summary\n\n\ndef factory(context, variant, seed=None):\n    shuffle_seed = context["config"]["seed"] if seed is None else seed\n\n    def load(split, size, shuffle):\n        # All trials use exactly the same ORIGINAL validation and test images.\n        root = context["roots"][variant if split == "train" else "original"]\n        ds = keras.utils.image_dataset_from_directory(\n            root / split,\n            class_names=context["classes"],\n            label_mode="categorical",\n            image_size=(size, size),\n            interpolation="bilinear",\n            batch_size=context["config"]["batch_size"],\n            shuffle=shuffle,\n            seed=shuffle_seed if shuffle else None,\n        )\n        paths = [Path(p).relative_to(context["extracted"]).as_posix() for p in ds.file_paths]\n        return ds.prefetch(tf.data.AUTOTUNE), paths\n\n    return load\n\n\ndef run(context):\n    c, output = context["config"], context["output"]\n    if c["mode"] == "comparison":\n        trials = [\n            dict(id=kind, backbone="B0", size=224, loss="ce", variant=kind)\n            for kind in ("original", "augmented")\n        ]\n    else:\n        trials = [dict(spec, variant=c["train_variant"]) for spec in engine.TRIALS]\n    records = []\n    try:\n        for spec in trials:\n            spec["class_count"] = len(context["classes"])\n            spec["class_names"] = context["classes"]\n            load = factory(context, spec["variant"])\n            record = engine.run_trial(\n                spec,\n                load,\n                output,\n                context["signature"],\n                c["seed"],\n                c["epochs1"],\n                0 if c["mode"] == "comparison" else c["epochs2"],\n            )\n            records.append(record)\n            engine.write_json(output / "progress.json", {"completed": [r["id"] for r in records]})\n        if c["mode"] == "suite":\n            parent = records[-1]\n            records.append(\n                engine.extend_b1(\n                    parent,\n                    factory(context, c["train_variant"]),\n                    output,\n                    context["signature"],\n                    c["seed"],\n                    c["extension_epochs"],\n                )\n            )\n        engine.write_json(output / "all_validation_results.json", records)\n        return records\n    except Exception as exc:\n        engine.write_json(\n            output / ("failure_" + uuid.uuid4().hex[:8] + ".json"),\n            {\n                "error": repr(exc),\n                "completed": [r["id"] for r in records],\n                "resume_dir": str(output),\n            },\n        )\n        print("중단. 완료된 실험을 유지합니다. RESUME_DIR:", output)\n        raise\n\n\ndef confusion(ax, metrics, title):\n    cm = np.asarray(metrics["confusion_matrix"])\n    ax.imshow(cm, cmap="Blues")\n    codes = [f"C{i}" for i in range(len(cm))]\n    ax.set(\n        title=title,\n        xlabel="Predicted",\n        ylabel="True",\n        xticks=range(len(cm)),\n        yticks=range(len(cm)),\n        xticklabels=codes,\n        yticklabels=codes,\n    )\n    for i in range(len(cm)):\n        for j in range(len(cm)):\n            ax.text(\n                j,\n                i,\n                str(cm[i, j]),\n                ha="center",\n                va="center",\n                fontsize=8,\n                color="white" if cm[i, j] > cm.max() / 2 else "black",\n            )\n\n\ndef errors(context, csv_path, destination):\n    import matplotlib.pyplot as plt\n    import pandas as pd\n    from PIL import Image\n\n    frame = pd.read_csv(csv_path)\n    wrong = frame[frame.true_index != frame.pred_index].head(8)\n    fig, axes = plt.subplots(2, 4, figsize=(14, 7))\n    for ax in axes.flat:\n        ax.axis("off")\n    for ax, (_, row) in zip(axes.flat, wrong.iterrows(), strict=False):\n        path = (context["extracted"] / row["path"]).resolve()\n        if not path.is_relative_to(context["extracted"].resolve()):\n            raise ValueError("Prediction path escapes dataset")\n        with Image.open(path) as image:\n            ax.imshow(image.convert("RGB"))\n        ax.set_title(f"True C{row.true_index} / Pred C{row.pred_index}")\n    if wrong.empty:\n        fig.suptitle("No misclassifications")\n    fig.tight_layout()\n    fig.savefig(destination, dpi=160)\n    plt.close(fig)\n\n\ndef overview(context, records):\n    import matplotlib.pyplot as plt\n    import pandas as pd\n\n    output = context["output"]\n    fig, axes = plt.subplots(\n        (len(records) + 1) // 2, 4, figsize=(24, 4.5 * ((len(records) + 1) // 2)), squeeze=False\n    )\n    for index, record in enumerate(records):\n        row, col = divmod(index, 2)\n        for offset, metric in enumerate(("accuracy", "loss")):\n            ax = axes[row, col * 2 + offset]\n            h = record["history"]\n            x = np.arange(1, len(h[metric]) + 1)\n            ax.plot(x, h[metric], label="Train")\n            ax.plot(x, h["val_" + metric], label="Validation")\n            if record["stage_boundary"] < len(x):\n                ax.axvline(record["stage_boundary"] + 0.5, ls="--", color="gray")\n            if "extension_boundary" in record:\n                ax.axvline(record["extension_boundary"] + 0.5, ls=":", color="green")\n            ax.set(title=record["id"] + " / " + metric, xlabel="Epoch", ylabel=metric)\n            if metric == "accuracy":\n                ax.set_ylim(0, 1)\n            ax.grid(alpha=0.25)\n            ax.legend()\n        directory = output / record["id"] / record["attempt"]\n        errors(\n            context, directory / "validation_predictions.csv", directory / "validation_errors.png"\n        )\n    fig.suptitle(context["config"]["domain"] + " / Loss definitions differ across CE, LS, Focal")\n    fig.tight_layout()\n    fig.savefig(output / "all_training_curves.png", dpi=180)\n    fig.savefig(output / "all_training_curves.pdf")\n    plt.show()\n    plt.close(fig)\n    table = pd.DataFrame(\n        [\n            dict(\n                experiment=r["id"],\n                selected_stage=r["selected_stage"],\n                validation_accuracy=r["validation"]["accuracy"],\n                validation_macro_f1=r["validation"]["macro_f1"],\n                parameters=r["parameters"],\n                seconds_this_trial=r["training_seconds"],\n                epochs=len(r["history"]["accuracy"]),\n                model_bytes=r["model_bytes"],\n            )\n            for r in records\n        ]\n    )\n    table.to_csv(output / "experiment_comparison.csv", index=False, encoding="utf-8-sig")\n    print(table.to_string(index=False))\n    fig, axes = plt.subplots(2, 1, figsize=(14, 11))\n    x = np.arange(len(records))\n    axes[0].bar(x - 0.2, table.validation_accuracy, 0.4, label="Validation Accuracy")\n    axes[0].bar(x + 0.2, table.validation_macro_f1, 0.4, label="Validation Macro F1")\n    axes[0].set(xticks=x, xticklabels=table.experiment, ylim=(0, 1))\n    axes[0].tick_params(axis="x", labelrotation=15)\n    axes[0].legend()\n    matrix = np.array([r["validation"]["class_f1"] for r in records])\n    axes[1].imshow(matrix, cmap="Blues", vmin=0, vmax=1, aspect="auto")\n    axes[1].set(\n        xticks=range(len(context["classes"])),\n        xticklabels=[f"C{i}" for i in range(len(context["classes"]))],\n        yticks=x,\n        yticklabels=table.experiment,\n        title="Validation class F1",\n    )\n    for i in range(len(records)):\n        for j in range(len(context["classes"])):\n            axes[1].text(j, i, str(matrix[i, j]), ha="center", va="center", fontsize=7)\n    fig.tight_layout()\n    fig.savefig(output / "validation_performance_dashboard.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n    fig, axes = plt.subplots(\n        (len(records) + 2) // 3, 3, figsize=(18, 6 * ((len(records) + 2) // 3)), squeeze=False\n    )\n    for ax in axes.flat:\n        ax.axis("off")\n    for ax, r in zip(axes.flat, records, strict=False):\n        ax.axis("on")\n        confusion(ax, r["validation"], r["id"])\n    fig.tight_layout()\n    fig.savefig(output / "all_validation_confusion_matrices.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n\n\ndef finish(context, records):\n    import matplotlib.pyplot as plt\n\n    output = context["output"]\n    overview(context, records)\n    winner = engine.select_winner(records)\n    model_path = engine.selected_model_path(output, winner)\n    selection = dict(\n        winner=winner["id"],\n        model_sha256=engine.file_hash(model_path),\n        signature=context["signature"],\n        validation=winner["validation"],\n    )\n    selected_file = output / "selection_before_test.json"\n    if selected_file.exists() and engine.read_json(selected_file) != selection:\n        raise ValueError("이미 고정한 선택 모델이 다릅니다.")\n    if not selected_file.exists():\n        engine.write_json(selected_file, selection)\n    marker = output / "test_completed.json"\n    if marker.exists():\n        tested = engine.read_json(marker)\n        if tested["selection"] != selection:\n            raise ValueError("기존 Test 모델과 다릅니다.")\n        for name, digest in tested["hashes"].items():\n            if engine.file_hash(output / name) != digest:\n                raise ValueError("Test 파일이 변경됐습니다.")\n        metrics = tested["metrics"]\n    else:\n        keras.backend.clear_session()\n        model = keras.models.load_model(model_path, compile=False)\n        ds, paths = factory(context, winner["spec"]["variant"])(\n            "test", winner["spec"]["size"], False\n        )\n        metrics = engine.evaluate_to_files(model, ds, paths, output, "final_test")\n        engine.write_json(\n            marker,\n            dict(\n                selection=selection,\n                metrics=metrics,\n                hashes={\n                    name: engine.file_hash(output / name)\n                    for name in ("final_test_metrics.json", "final_test_predictions.csv")\n                },\n            ),\n        )\n        del model\n    fig, ax = plt.subplots(figsize=(8, 8))\n    confusion(ax, metrics, winner["id"] + " / Final Test")\n    fig.tight_layout()\n    fig.savefig(output / "final_test_confusion_matrix.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n    errors(context, output / "final_test_predictions.csv", output / "final_test_errors.png")\n    card = dict(\n        domain=context["config"]["domain"],\n        class_names=context["classes"],\n        normal_included="정상" in context["classes"],\n        validation=winner["validation"],\n        test=metrics,\n        selected_model=winner["id"],\n        model_sha256=selection["model_sha256"],\n        input_size=winner["spec"]["size"],\n        data_sha256=context["data_hash"],\n        status="public_data_candidate_not_device_validated",\n        limitations=context["audit"]["limitations"]\n        + [\n            "Single seed; small differences are not established as robust gains",\n            "Out-of-scope rejection absent; scores are not calibrated correctness",\n        ],\n    )\n    engine.write_json(output / "model_card.json", card)\n    package_parent = (\n        context["project"] / "2_results" / context["config"]["domain"] / "selected_models"\n    )\n    package = package_parent / (output.name + "_" + uuid.uuid4().hex[:8])\n    package.mkdir(parents=True, exist_ok=False)\n    shutil.copyfile(model_path, package / "model.keras")\n    if engine.file_hash(package / "model.keras") != selection["model_sha256"]:\n        raise OSError("모델 복사 불일치")\n    for name in (\n        "class_names.json",\n        "model_card.json",\n        "run_config.json",\n        "selection_before_test.json",\n        "audit_summary.json",\n        "final_test_metrics.json",\n        "common_engine.py",\n        "common_audit.py",\n        "common_workflow.py",\n    ):\n        shutil.copyfile(output / name, package / name)\n    engine.write_json(\n        package / "preprocessing.json",\n        dict(\n            input_shape=[winner["spec"]["size"], winner["spec"]["size"], 3],\n            color="RGB",\n            dtype="float32",\n            pixel_range=[0, 255],\n            external_normalization=False,\n            internal_rescaling="1/255",\n            resize="TensorFlow bilinear; no crop/pad; antialias=False",\n            exif_transpose=False,\n            output="softmax scores in class_names.json order",\n        ),\n    )\n    engine.write_json(\n        package / "manifest.json",\n        {p.name: engine.file_hash(p) for p in package.iterdir() if p.is_file()},\n    )\n    destination = Path(shutil.make_archive(str(package), "zip", package.parent, package.name))\n    with zipfile.ZipFile(destination) as archive:\n        if archive.testzip():\n            raise OSError("모델 ZIP 손상")\n        manifest = engine.read_json(package / "manifest.json")\n        for name, digest in manifest.items():\n            if hashlib.sha256(archive.read(package.name + "/" + name)).hexdigest() != digest:\n                raise OSError("ZIP 내용 불일치: " + name)\n    destination.with_suffix(".zip.sha256").write_text(\n        engine.file_hash(destination), encoding="ascii"\n    )\n    archive_results(context)\n    print("선정 모델:", winner["id"], "\\nTest:", metrics, "\\n후보 ZIP:", destination)\n    return card\n'}
BUILD_COMMIT = '268e2ad22fd23ca37d6a2987ef391944b9c7fd1b'
PROFILES = {'hair': ['모낭사이홍반', '미세각질', '비듬', '탈모', '피지과다'], 'web_skin': ['건선', '아토피', '여드름', '정상', '주사'], 'skin': ['광선각화증', '기저세포암', '보웬병', '사마귀', '지루각화증', '편평세포암', '표피낭종', '피부섬유종', '혈관종', '흑색점']}
package = types.ModuleType('mediflow_datasets')
package.__path__ = []
sys.modules['mediflow_datasets'] = package
for name, source in SOURCES.items():
    module = types.ModuleType('mediflow_datasets.' + name)
    sys.modules[module.__name__] = module
    exec(compile(source, name + '.py', 'exec'), module.__dict__)
from mediflow_datasets.common_workflow import prepare, run, finish, audit_run


## 4. 데이터 준비와 실험 기록

Drive ZIP을 Colab 임시 디스크로 복사하고 해시와 안전한 압축 경로, 여유 공간을 확인합니다.
ZIP 내부는 `original/train/클래스`, `original/val/클래스`, `original/test/클래스`와
동일한 `augmented/...` 구조여야 합니다. Original/Augmented의 대소문자는 허용합니다.
예상 클래스와 다르면 자동으로 이름을 추측하지 않고 중단합니다.

학습 노트북은 공통 ① 보고서 또는 사용자가 입력한 SHA-256과 ZIP이 같은지 확인합니다.
SHA-256 방식은 파일이 바뀌지 않았다는 것만 확인하며 독립 데이터 감사를 대신하지 않습니다.
기계적 검사를 통과해도 사람·병변·촬영 세션·변형된 증강 파생본 누수가 없다는 뜻은 아닙니다.
현재 출처 대응 정보가 없어 해당 항목은 미검증으로 기록합니다.


In [ ]:
config = dict(domain=DOMAIN, project_root=PROJECT_ROOT, data_zip=DATA_ZIP,
              audit_dir=AUDIT_DIR, expected_data_sha256=EXPECTED_DATA_SHA256,
              resume_dir=RESUME_DIR, mode=MODE, seed=SEED,
              batch_size=BATCH_SIZE, epochs1=STAGE1_EPOCHS, epochs2=STAGE2_EPOCHS,
              extension_epochs=EXTENSION_EPOCHS, train_variant=TRAIN_VARIANT)
context = prepare(config, PROFILES, SOURCES, BUILD_COMMIT)
print('결과 폴더 / 중단 시 RESUME_DIR:', context['output'])
print('클래스 순서:', context['classes'])


## 5. 이미지 검사와 보고서 저장

손상·지원하지 않는 형식·클래스 개수·빈 클래스·파일/픽셀 중복·다른 라벨의 동일 사진을 검사합니다.
원본과 증강본의 검증/테스트가 같은지도 확인합니다. 중복 예시 그림과 개수 그래프를 저장합니다.
문제가 있으면 학습은 차단됩니다. 자동으로 사진을 삭제하거나 분할을 변경하지 않습니다.
`audit_summary.json`의 status와 limitations를 읽고, 출력된 폴더를 ②/③의 AUDIT_DIR에 넣으세요.


In [ ]:
audit_run(context)
